# 02 — Peak initialisation

This notebook confirms the peak-based initialisation that seeds the fit. It reproduces the logic of `PeakInitialiser` from `pipelines.param_pipeline.sigma`: a width-5 boxcar smoothing followed by prominence-based `scipy.signal.find_peaks`, selecting the `K` most prominent peaks as initial centroids and amplitudes.

Modules exercised (reproduced on CPU):

- `pipelines.param_pipeline.sigma.PeakInitialiser` (algorithm reproduced inline; the production module imports JAX, which is not available in this environment)
- `scipy.ndimage.uniform_filter1d`, `scipy.signal.find_peaks`

Profiles have known peak locations so the recovered centroids can be checked by eye.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


## Reproduce the initialiser worker

The function below mirrors `PeakInitialiser._prominence_worker` and the `sigma_guess` / `min_dist` heuristics from `PeakInitialiser.run`. The production class wraps this in a `ProcessPoolExecutor`; the math is identical.

In [ ]:
from scipy.ndimage import uniform_filter1d
from scipy.signal  import find_peaks

def peak_init(prof_raw, height_axis, K, prominence_frac=0.05):
    N, H        = prof_raw.shape
    h_span      = float(height_axis[-1] - height_axis[0])
    dh          = float(height_axis[1] - height_axis[0])
    sigma_guess = max(2.0 * dh, h_span / (8.0 * K))
    min_dist    = max(1, int(sigma_guess / dh))
    smoothed    = uniform_filter1d(prof_raw.astype(np.float32), size=5, mode='nearest', axis=1)

    amps = np.zeros((N, K), dtype=np.float32)
    mus  = np.zeros((N, K), dtype=np.float32)
    sigs = np.full ((N, K), sigma_guess, dtype=np.float32)

    for n in range(N):
        smo  = smoothed[n]
        pmax = smo.max()
        if pmax < 1e-10:
            idxs = np.linspace(0, H - 1, K, dtype=int)
        else:
            peaks, props = find_peaks(smo, prominence=pmax * prominence_frac, distance=min_dist)
            if len(peaks) >= K:
                idxs = peaks[np.argsort(props['prominences'])[::-1][:K]]
            elif len(peaks) > 0:
                residual        = smo.copy()
                residual[peaks] = 0.0
                extra           = []
                for _ in range(K - len(peaks)):
                    ei = int(np.argmax(residual))
                    extra.append(ei)
                    lo = max(0, ei - min_dist)
                    hi = min(H, ei + min_dist + 1)
                    residual[lo:hi] = 0.0
                idxs = np.concatenate([peaks, np.array(extra, dtype=int)])
            else:
                idxs = np.linspace(0, H - 1, K, dtype=int)
        for g, idx in enumerate(idxs[:K]):
            amps[n, g] = max(float(smo[idx]), 1e-10)
            mus [n, g] = float(height_axis[idx])
    return amps, mus, sigs

print('initialiser reproduced')

## Build profiles with known peaks

We construct three noisy profiles whose true centroids are known, then run the initialiser and compare the recovered centroids against the truth.

In [ ]:
K           = 3
H           = 180
height_axis = np.linspace(0.0, 45.0, H).astype(np.float32)

truth_sets = [
    [(1.0,  8.0, 2.0), (0.7, 24.0, 3.0), (0.4, 36.0, 1.8)],
    [(0.9, 12.0, 2.5), (0.9, 30.0, 2.5), (0.0,  0.0, 1.0)],
    [(1.0, 20.0, 4.0), (0.0,  0.0, 1.0), (0.0,  0.0, 1.0)],
]

profiles = np.stack([
    gaussian_mixture_profile(height_axis, [c for c in s if c[0] > 0], noise_std=0.03, rng=rng)
    for s in truth_sets
])

amps0, mus0, sigs0 = peak_init(profiles, height_axis, K)
print('init centroids (mu) per profile:')
print(np.round(mus0, 2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for n, ax in enumerate(axes):
    ax.plot(height_axis, profiles[n], color='black', lw=1.2, label='profile')
    true_mus = [c[1] for c in truth_sets[n] if c[0] > 0]
    for tm in true_mus:
        ax.axvline(tm, color='tab:green', lw=1.4, ls='-', alpha=0.8)
    for k in range(K):
        if amps0[n, k] > 1e-6:
            ax.axvline(mus0[n, k], color='tab:red', lw=1.1, ls='--', alpha=0.9)
    ax.set_title(f'profile {n + 1}')
    ax.set_xlabel('height h [m]')
axes[0].set_ylabel('intensity')
axes[0].plot([], [], color='tab:green', label='true centroid')
axes[0].plot([], [], color='tab:red', ls='--', label='init centroid')
axes[0].legend(fontsize=8)
fig.suptitle('Peak initialisation: recovered vs true centroids')
fig.tight_layout()
plt.show()

## Expected outcome

For each profile the dashed red initial centroids should fall on or very close to the solid green true centroids of the genuine peaks. Where a profile has fewer real peaks than `K`, the surplus slots are placed at the next-largest residual locations, demonstrating the fallback branch of the initialiser without disturbing the recovered true peaks.